# **Journalism Benchmark Cookbook: Information Extraction**
In this notebook, we demonstrate how to evaluate an information extraction task based on a scenario with **textual information extraction** from **structured** **image documents**.

This notebook can be copied and modified to fit other similar scenarios of evaluation. For other specific tasks in information extraction, see [here](https://github.com/shallotly/news-eval-cookbook).

# Evaluation Overview

*   **What is the use scenario?** In this use case, we aim to test the *information extraction* abilities of generative AI using a scenario in [the Markup's reporting on Banned items on Amazon](https://themarkup.org/banned-bounty/2020/06/18/amazons-enforcement-failures-leave-open-a-back-door). This story focuses on listings of banned items on Amazon's marketplace and examines how well Amazon enforces its policies for listing banned items.

*   **What is the dataset?** The dataset can be accessed at: https://github.com/the-markup/investigation-amazon-banned-items/blob/master/data/amazon-banned-items-findings.csv. The dataset consists of 97 screenshots of banned item listings on Amazon, which The Markup reporters searched for and collected via specific keywords, For each screenshot, they also list the related text in Amazon's own [restricted product page](https://sellercentral.amazon.com/help/hub/reference/external/200164330). [Below](#data), we provide a brief explanation of data fields.

*   **How is the test case set up?**
We employ generative AI to extract information regarding whether an item is listed under restricted products, the category of the restricted product, and the policy text related to the restricted product given the product page screenshot and an url to Amazon's restricted product policy page. <br> We prompt systems with a text prompt: *Given the following screenshot of a product on amazon, please identify if the product listing violates any categories on Amazon's restricted product page (https://sellercentral.amazon.com/help/hub/reference/external/200164330).
If the item in the image is listed under restricted products, please identify the product category under which it is listed and navigate to the specific page where the restriction policy is listed in text.
Extract the original restriction text from the page regarding the item.* We also provide a structured response format for model outputs.

*   **How are we assessing the performance of gen AI?**
We calculate an accuracy score based whether a model correctly identifies an image as a restricted product. We also calculate the longest common substring for both the product category and restricted product policy text identified by the model in comparison to the human identified ground truth for accuracy.


<a name="data"></a>
# 1. Data

The dataset contains the following relevant fields :

  * **restricted_product_category**: The Restricted Products category that the prohibited product falls under.(e.g. "Drugs & drug paraphernalia")
  * **prohibited_policy_text**: The specific text of the prohibition from the Restricted Products policy page. Bulleted list items include the text at the top of the list. (e.g. "Listing for injectable drugs are prohibited")
  * **policy_page**: A link to a timestamped screenshot of the Restricted Products policy page that covers this prohibition.
  * **item_type**: Description of the item. (e.g. "Compounds reviewers were using as injectable drugs")
  * **asin**: The Amazon Standard Identification Number (ASIN) is an alphanumeric unique identifier for products in Amazon's catalog.
  * **screenshot**: A link to a full page screenshot of the product listing.
  * **listing_url**: 	The URL of the product listing that we found.
  

In total, there are 97 rows, each corresponding to one product, in the dataset.

## 1.1 Data Code
The following code loads the data from github into a python list of dictionaries. It is further filtered and truncated.

In [ ]:
!curl -O https://raw.githubusercontent.com/the-markup/investigation-amazon-banned-items/refs/heads/master/data/amazon-banned-items-findings.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 58238  100 58238    0     0   203k      0 --:--:-- --:--:-- --:--:--  204k


In [ ]:
import csv

data = []
with open('amazon-banned-items-findings.csv', 'r', encoding='utf-8') as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for row in csv_reader:
        data.append(row)

# 2. Task Details
The scenario above outlines an *information extraction task*. This task is for the AI model to correctly extract specific information or excerpts from images along with textual information on the web.

This scenario fills a specific journalistic task context in that it is an *internal task*, which means the output of this task is used by internal newsroom staff, and it contains *strutured sources* in the way that product information pages have specific layouts, but also it is *unstructured* within the policy texts. This scenario focuses on data that is image and that is stored in CSV file.

We summarize the task context here:

| Task Context | Value |
| :----------- | :---- |
| Task Usage   | Internal Usage (not public facing) |
| File Format  | csv |
| Modality     | Image |
| Data Structure | Structured |

## 2.1 Task Code   
The following code blocks first define a textual prompt input for models and then uses the [OpenRouter API](https://openrouter.ai/docs/quickstart#using-the-openrouter-api-directly) to collect model responses.

A prompt is input with each document for 4 different models (`openai/gpt-5`, `anthropic/claude-sonnet-4.5`, `qwen/qwen2.5-vl-32b-instruct`, `meta-llama/llama-4-maverick`). We specifically chose models that contain image processing capabilities (https://openrouter.ai/models?fmt=cards&input_modalities=image) The code provides a template for [structured output](https://openrouter.ai/docs/features/structured-outputs).

To authenticate the Open Router API with your own API key, refer to [this](https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb#scrollTo=yeadDkMiISin) to create a separate name-value pair for Open Router.

The last two code blocks loads model outputs as python `dict` for access in the next portion and exports outputs as a json file.

In [ ]:
prompt = """Given the following screenshot of a product on amazon, please identify if the product listing violates any categories on Amazon's restricted product page (https://sellercentral.amazon.com/help/hub/reference/external/200164330).
If the item in the image is listed under restricted products, please identify the product category under which it is listed and navigate to the specific page where the restriction policy is listed in text.
Extract the original restriction text from the page regarding the item. Provide the response in the following json format: {'violation_to_restriction':(boolean), 'restricted_product_category': (string), 'prohibited_policy_text': (string)}"""

In [3]:
models = ["openai/gpt-5", "anthropic/claude-sonnet-4.5", "qwen/qwen2.5-vl-32b-instruct", "meta-llama/llama-4-maverick"] #, "google/gemini-2.5-flash-image",

In [ ]:
import requests

def download_image(image_url):
  save_path = image_url.rsplit('/', 1)[1]  # Desired filename for the saved image
  try:
      response = requests.get(image_url, stream=True)
      response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)
      with open(save_path, 'wb') as file:
          for chunk in response.iter_content(chunk_size=8192):
              file.write(chunk)
  except requests.exceptions.RequestException as e:
      print(f"Error downloading image: {e}")

In [ ]:
from PIL import Image

def resize_image_pillow(image_path):
    try:
        with Image.open(image_path) as img:
            size = img.size
            new_width = 1280
            new_height = int(1280/size[0]*size[1])
            resized_img = img.resize((new_width, new_height), Image.LANCZOS) # LANCZOS for high-quality downsampling
            resized_img.save(image_path)
        print(f"Image resized and saved to {image_path}")
    except FileNotFoundError:
        print(f"Error: Image not found at {image_path}")
    except Exception as e:
        print(f"Error resizing image: {e}")

In [ ]:
import tqdm

for task in tqdm.tqdm(data):
    image_path = task['screenshot'].rsplit('/', 1)[1]
    #download_image(task['screenshot'])
    resize_image_pillow(image_path)

In [ ]:
import base64
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [ ]:
from google.colab import userdata
import gc
import json

url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {userdata.get('openrouter_api')}",
    "Content-Type": "application/json"
}

response_format ={
    "type": "json_schema",
    "json_schema": {
      "name": "restricted_item",
      "strict": True,
      "schema": {
        "type": "object",
        "properties": {
          "violation_to_restriction": {
            "type": "boolean",
            "description": "whether product listing violates amazon's restricted products policy"
          },
          "restricted_product_category": {
            "type": "string",
            "description": "category of restricted product the item falls under"
          },
          "prohibited_policy_text": {
            "type": "string",
            "description": "policy text regarding restrictions for the item"
          }
        },
        "required": ["violation_to_restriction", "restricted_product_category", "prohibited_policy_text", ],
        "additionalProperties": False
      }
    }
}

errors_indices = {model:[] for model in models}
f = open('error.log','w')
f.close()

for model in models:
  print(f"currently prompting {model}")
  for task in tqdm.tqdm(data):
    image_path = task['screenshot'].rsplit('/', 1)[1]
    image_bits = encode_image_to_base64(image_path)
    messages = [
      {
          "role": "user",
          "content": [
              {
                  "type": "text",
                  "text": prompt
              },
              {
                  "type": "image_url",
                  "image_url": {
                      "url": f"data:image/png;base64,{image_bits}"
                  }
              }
          ]
      }
    ]

    payload = {
        "model": model,
        "messages": messages,
        "response_format": response_format,
    }

    response = requests.post(url, headers=headers, json=payload)
    try:
      raw_response = response.json()
      task[model+'_output'] = raw_response['choices'][0]['message']['content']
    except KeyError:
      print(f'Key error for model: {model} with task #{data.index(task)}.')
      errors_indices[model].append(data.index(task))
      with open('error.log','a') as f:
        f.write(f'Key error for model: {model} with task #{data.index(task)}. Model response: {response.json()}\n')
    except requests.JSONDecodeError:
      print(f'JSONDecode error for model: {model} with task #{data.index(task)}.')
      errors_indices[model].append(data.index(task))
      with open('error.log','a') as f:
        f.write(f'Key error for model: {model} with task #{data.index(task)}. Model response: {response}\n')

    #delete downloaded image to save memory
    #os.remove(image_path)
    del image_bits
    gc.collect()
  with open(f'output_10-16-{models.index(model)}.json', 'w') as f:
    json.dump(data,f)
with open('error.log','a') as f:
  json.dump(errors_indices,f)

In [1]:
import json

with open('output-10-17.json','r') as f:
  data = json.load(f)

In [52]:
import ast
import requests
import re

for model in models:
  for task in data:
    try:
      if task[model+'_output'].startswith("{") and task[model+'_output'].endswith("}"):
        task[model+"_json"] = json.loads(task[model+'_output'])
      else:
        try:
          matches = re.findall(r"\{(.*?)\}", task[model+'_output'], re.S)
          task[model+"_json"] = json.loads('{'+matches[0]+'}')
        except IndexError:
          print(f"index error: {model}, {data.index(task)}")
          task[model+"_json"] = {}
    except json.JSONDecodeError:
      print(f"jsondecode error: {model}, {data.index(task)}")
      matches = re.findall(r"\{(.*?)\}", task[model+'_output'], re.S)
      s = '{'+matches[0]+'}'
      s = s.replace("false","False")
      s = s.replace("true","True")
      print(s)
      task[model+"_json"] = eval(s)




index error: anthropic/claude-sonnet-4.5, 0
index error: anthropic/claude-sonnet-4.5, 5
jsondecode error: anthropic/claude-sonnet-4.5, 16
{'violation_to_restriction': False, 'restricted_product_category': 'N/A', 'prohibited_policy_text': 'N/A'}
jsondecode error: anthropic/claude-sonnet-4.5, 28
{'violation_to_restriction': True, 'restricted_product_category': 'Drugs & Drug Paraphernalia', 'prohibited_policy_text': 'Listings that promote, suggest, or facilitate the use, sale, or distribution of illegal drugs or controlled substances are prohibited. This includes products marketed or intended for the use of illegal drugs, controlled substances, or drug paraphernalia. Products that can be used with illegal drugs, controlled substances, or tobacco products are prohibited. Products marketed as "for tobacco use only" or similar disclaimers are prohibited.'}
jsondecode error: anthropic/claude-sonnet-4.5, 34
{'violation_to_restriction': True, 'restricted_product_category': 'Knives, Swords, and 

# 3. Evaluation Metrics
We focus on evaluating the *accuracy* of the models against information identified by reporters in the dataset.

*   We calculate the accuracy of models on extracting information about whether a product is a prohibited item based on the screenshot of product information page provided, as well as the accuracy of extracting prohibited product category and restriction policy text for the product. For accuracy of extracting information about whether the item is prohibited, we calculate the number of items that a model identifies as prohibited (since all products are prohibited in the dataset). The accuracy values range from 0 to 1. For the accuracy of identifying product category and policy texts, we measure the longest common substring between the model generated answer and the text provided by the reporters, we then average the portion of the "original string" a model is able to match with its generated text, hence this value also range between 0 and 1.

#### Other Potential Metrics
Other factors that could be measured includes speed, which would rely on measuring how fast the model produced a response, or whether a listing is since been taken down, but this metric is not implemented here. Additionally, uncertainty in terms of information extracted could also be measured by either prompting the model for its certainty or prompting it multiple times with the same input and aggregate for agreement.


# 4. Performance Measurments

#    
The following code blocks calculate the accuracy score for each models on each of the variables in the generated data (`violation_to_restriction`, `prohibited_policy_text` and `restricted_product_category`). It is worth noting that the anthropic model did not generate a json response for a few data points.

In [15]:
from prettytable import PrettyTable

In [16]:
data[1]

{'restricted_product_category': 'Drugs & drug paraphernalia',
 'prohibited_policy_text': 'Damiana',
 'policy_page': 'https://github.com/the-markup/investigation-amazon-banned-items-screenshots/raw/master/screenshots/Drugs-drug-paraphernalia-screencapture-sellercentral-amazon-gp-help-external-help-html-2020-01-21-12_46_43.png',
 'item_type': 'Damiana',
 'asin': 'B001E131DC',
 'screenshot': 'https://github.com/the-markup/investigation-amazon-banned-items-screenshots/raw/master/screenshots/B001E131DC-2020-02-08-09_57_23.png',
 'listing_url': 'https://www.amazon.com/Natures-Way-Damiana-Caps-Pack/dp/B001E131DC',
 'openai/gpt-5_output': '{"violation_to_restriction":false,"restricted_product_category":"","prohibited_policy_text":""}',
 'anthropic/claude-sonnet-4.5_output': '```json\n{\n  "violation_to_restriction": true,\n  "restricted_product_category": "Dietary Supplements",\n  "prohibited_policy_text": "Listings for dietary supplements must comply with applicable laws and regulations, incl

In [53]:
#calculate probihited product identification accuracy

t = PrettyTable(['Model', 'Correct #', 'Accuracy'])

for model in models:
  model_true = 0
  for task in data:
    try:
      if (type(task[model+'_json'])==type('string')):
        task[model+'_json'] = json.loads(task[model+'_json'])
      if task[model+'_json']['violation_to_restriction'] == True:
        model_true += 1
    except KeyError:
      print(f"key error: {model}, {task[model+'_json']}")
  t.add_row([model, model_true, model_true/len(data)])

print(f"The following table shows each model's accuracy score on 97 products reporters identified as prohibited")
print(t)

key error: anthropic/claude-sonnet-4.5, {}
key error: anthropic/claude-sonnet-4.5, {}
The following table shows each model's accuracy score on 97 products reporters identified as prohibited
+------------------------------+-----------+---------------------+
|            Model             | Correct # |       Accuracy      |
+------------------------------+-----------+---------------------+
|         openai/gpt-5         |     51    |  0.5257731958762887 |
| anthropic/claude-sonnet-4.5  |     37    | 0.38144329896907214 |
| qwen/qwen2.5-vl-32b-instruct |     58    |  0.5979381443298969 |
| meta-llama/llama-4-maverick  |     62    |  0.6391752577319587 |
+------------------------------+-----------+---------------------+


In [56]:
from difflib import SequenceMatcher

#measure prohibited product policy text accuracy
#code adapted from google
def find_longest_common_substring(groundtruth, generated):
    matcher = SequenceMatcher(None, groundtruth, generated)
    overlap = matcher.find_longest_match(0, len(groundtruth), 0, len(generated))

    if overlap.size == 0:
        return 0
    else:
        return overlap.size/len(groundtruth)

t = PrettyTable(['Model', 'Score'])
for model in models:
  score = 0
  groundtruth = 0
  for task in data:
    try:
      groundtruth_string = task['prohibited_policy_text']
      generated_string = task[model+'_json']['prohibited_policy_text']
      if type(groundtruth_string) == str:
        groundtruth += 1
        if type(generated_string) == str:
          score += find_longest_common_substring(groundtruth_string, generated_string)
    except KeyError:
      score += 0
  t.add_row([model,score/groundtruth])

print(f"The following table shows each model's accuracy score on {groundtruth} groundtruth answers for identifying 'policy text'")
print(t)

The following table shows each model's accuracy score on 97 groundtruth answers for identifying 'policy text'
+------------------------------+----------------------+
|            Model             |        Score         |
+------------------------------+----------------------+
|         openai/gpt-5         | 0.07101834141535081  |
| anthropic/claude-sonnet-4.5  | 0.029449155109930323 |
| qwen/qwen2.5-vl-32b-instruct | 0.03704678496756351  |
| meta-llama/llama-4-maverick  | 0.04492021899205211  |
+------------------------------+----------------------+


In [57]:
#measure prohibited product category accuracy
#code adapted from google
def find_longest_common_substring(groundtruth, generated):
    matcher = SequenceMatcher(None, groundtruth, generated)
    overlap = matcher.find_longest_match(0, len(groundtruth), 0, len(generated))

    if overlap.size == 0:
        return 0
    else:
        return overlap.size/len(groundtruth)

t = PrettyTable(['Model', 'Score'])
for model in models:
  score = 0
  groundtruth = 0
  for task in data:
    try:
      groundtruth_string = task['restricted_product_category']
      generated_string = task[model+'_json']['restricted_product_category']
      if type(groundtruth_string) == str:
        groundtruth += 1
        if type(generated_string) == str:
          score += find_longest_common_substring(groundtruth_string, generated_string)
    except KeyError:
      score += 0
  t.add_row([model,score/groundtruth])

print(f"The following table shows each model's accuracy score on {groundtruth} groundtruth answers for identifying 'product category'")
print(t)

The following table shows each model's accuracy score on 97 groundtruth answers for identifying 'product category'
+------------------------------+---------------------+
|            Model             |        Score        |
+------------------------------+---------------------+
|         openai/gpt-5         | 0.33047293044112325 |
| anthropic/claude-sonnet-4.5  | 0.07721431305345716 |
| qwen/qwen2.5-vl-32b-instruct | 0.11186657468488068 |
| meta-llama/llama-4-maverick  | 0.12892241816222585 |
+------------------------------+---------------------+
